In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import glob
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def crpic(points, positions, hauteur, largeur=0.02):
    return hauteur * (largeur**2) / ((points - positions)**2 + largeur**2)

def normaliser_intensites(df):
    max_int = df['Int.'].max()
    df['Int.'] = df['Int.'] / max_int * 1000
    return df

points_13C = np.linspace(0, 200, 30000)

In [3]:
SOLVANTS_13C = {
    'cdcl3':   {'ppm': 77.16, 'Int.': 500},
    'ccl4':    None,
    'dmso-d6': {'ppm': 39.52, 'Int.': 700},
    'd2o':     None,
}

molecules_13C = {}
for dossier in glob.glob('/content/drive/MyDrive/IA RMN/Molecules CSV 13C/*/'):
    nom_solvant = Path(dossier).name.lower()
    pic_solvant = SOLVANTS_13C.get(nom_solvant, None)
    for fichier in glob.glob(f'{dossier}*.csv'):
        nom = Path(fichier).stem.split('(')[0]
        try:
            df = pd.read_csv(fichier)[['ppm', 'Int.']]
            df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
            df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
            df.dropna(inplace=True)
            df = normaliser_intensites(df)
            molecules_13C[nom] = {'df': df, 'solvant': pic_solvant}
        except Exception as e:
            print(f"Erreur {fichier}: {e}")

# Annotations
df_annotations = pd.read_csv('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/annotations_fonctions.csv',
                              index_col='molecule')

with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/pos_weights.json') as f:
    poids_data = json.load(f)
colonnes_fonctions = poids_data['colonnes']
n_fonctions = len(colonnes_fonctions)

# Rounded weights
freq = df_annotations[colonnes_fonctions].mean().values
pos_weights = np.sqrt((1 - freq) / freq).astype(np.float32)

noms_communs = set(molecules_13C.keys()) & set(df_annotations.index)
print(f"Molécules 13C       : {len(molecules_13C)}")
print(f"Molécules annotées  : {len(df_annotations)}")
print(f"13C avec annotation : {len(noms_communs)}")
print(f"Fonctions           : {n_fonctions}")

Molécules 13C       : 1244
Molécules annotées  : 1231
13C avec annotation : 1103
Fonctions           : 15


In [4]:
class SpectreGeneratorMultiLabel13C(tf.keras.utils.Sequence):
    def __init__(self, molecules_dict, df_annotations, colonnes_fonctions,
                 points, batch_size=128, n_per_molecule=50, augmentation=True,
                 workers=4, use_multiprocessing=True, max_queue_size=20, **kwargs):
        super().__init__(workers=workers, use_multiprocessing=use_multiprocessing,
                         max_queue_size=max_queue_size, **kwargs)
        self.molecules = [m for m in molecules_dict.keys() if m in df_annotations.index]
        self.points = points
        self.batch_size = batch_size
        self.augmentation = augmentation
        self.total = len(self.molecules) * n_per_molecule
        self.annotations = df_annotations[colonnes_fonctions]
        self.arrays = {m: (molecules_dict[m]['df']['ppm'].values,
                           molecules_dict[m]['df']['Int.'].values,
                           molecules_dict[m]['solvant']) for m in self.molecules}
        self.labels = {m: self.annotations.loc[m].values.astype(np.float32)
                       for m in self.molecules}
        print(f"Molécules avec annotations : {len(self.molecules)}")

    def __len__(self):
        return self.total // self.batch_size

    def __getitem__(self, idx):
        X = np.empty((self.batch_size, len(self.points), 1), dtype=np.float32)
        Y = np.empty((self.batch_size, len(self.annotations.columns)), dtype=np.float32)

        for i in range(self.batch_size):
            nom = np.random.choice(self.molecules)
            ppm_arr, int_arr, pic_solvant = self.arrays[nom]

            if self.augmentation:
                X_rand = np.random.uniform(-1.0, 1.0)
                Y_rand = np.random.uniform(0.8, 1.2)
                largeur = np.random.uniform(0.015, 0.025)
            else:
                X_rand, Y_rand, largeur = 0.0, 1.0, 0.02

            spectre = np.zeros_like(self.points)
            for p, h in zip(ppm_arr, int_arr):
                spectre += crpic(self.points, p + X_rand, h * Y_rand, largeur)

            if pic_solvant is not None and self.augmentation and np.random.random() < 0.7:
                spectre += crpic(self.points,
                                 pic_solvant['ppm'] + np.random.uniform(-0.2, 0.2),
                                 pic_solvant['Int.'] * np.random.uniform(0.8, 1.2),
                                 np.random.uniform(0.015, 0.025))

            if self.augmentation:
                amax = np.max(spectre)
                snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
                spectre += np.random.normal(0, amax / snr if amax > 0 else 0.01, len(self.points))

            X[i, :, 0] = spectre
            Y[i] = self.labels[nom]

        return X, Y

In [ ]:
import gc

np.random.seed(42)
gen_val_temp = SpectreGeneratorMultiLabel13C(
    molecules_13C, df_annotations, colonnes_fonctions,
    points_13C, batch_size=128, n_per_molecule=10, augmentation=True,
    workers=1, use_multiprocessing=False
)
X_val_list, Y_val_list = [], []
for i in range(len(gen_val_temp)):
    xb, yb = gen_val_temp[i]
    X_val_list.append(xb); Y_val_list.append(yb)
X_val_ml = np.concatenate(X_val_list, axis=0)
Y_val_ml = np.concatenate(Y_val_list, axis=0)
del X_val_list, Y_val_list, gen_val_temp; gc.collect()
print(f"Validation : {X_val_ml.shape}, labels : {Y_val_ml.shape}")

Molécules avec annotations : 1103
Validation : (11008, 30000, 1), labels : (11008, 15)


In [ ]:
model_13C = tf.keras.models.load_model(
    '/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/2_modele_RMN13C.h5'
)
dummy = tf.zeros((1, 30000, 1))
_ = model_13C(dummy)

# Diagnostic : repérer la couche Dense(64)
print("=== Couches ===")
for i, layer in enumerate(model_13C.layers):
    try:
        shape = layer.output.shape
    except Exception:
        shape = "?"
    print(f"  [{i}]  {layer.name:<25} {type(layer).__name__:<20} → {shape}")

=== Couches ===
  [0]  conv1d_3                  Conv1D               → (None, 7496, 32)
  [1]  batch_normalization_3     BatchNormalization   → (None, 7496, 32)
  [2]  max_pooling1d_3           MaxPooling1D         → (None, 1874, 32)
  [3]  conv1d_4                  Conv1D               → (None, 1870, 64)
  [4]  batch_normalization_4     BatchNormalization   → (None, 1870, 64)
  [5]  max_pooling1d_4           MaxPooling1D         → (None, 467, 64)
  [6]  conv1d_5                  Conv1D               → (None, 463, 128)
  [7]  batch_normalization_5     BatchNormalization   → (None, 463, 128)
  [8]  max_pooling1d_5           MaxPooling1D         → (None, 115, 128)
  [9]  average_pooling1d_1       AveragePooling1D     → (None, 14, 128)
  [10]  flatten_1                 Flatten              → (None, 1792)
  [11]  dense_2                   Dense                → (None, 64)
  [12]  dropout_1                 Dropout              → (None, 64)
  [13]  dense_3                   Dense           

In [ ]:
k = 11 # Index of the Dense(64) layer, found from the summary above

feature_extractor = tf.keras.Sequential(model_13C.layers[:k+1])
feature_extractor.build((None, 30000, 1))
feature_extractor.trainable = False

test_out = feature_extractor(dummy)
print(f"Sortie feature_extractor : {test_out.shape}") # must display (1, 64)

Sortie feature_extractor : (1, 64)


In [ ]:
inputs = tf.keras.Input(shape=(30000, 1))
x = feature_extractor(inputs, training=False)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(n_fonctions, activation='sigmoid', name='fonctions')(x)

model_ml = tf.keras.Model(inputs=inputs, outputs=output)
model_ml.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 30000, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 64)             │       167,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fonctions (Dense)               │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 170,287 (665.18 KB)

 Trainable params: 2,575 (10.06 KB)

 Non-trainable params: 167,712 (655.12 KB)

In [ ]:
def weighted_bce(pos_weights):
    w = tf.constant(pos_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -(w * y_true * tf.math.log(y_pred)
                + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss

model_ml.compile(
    optimizer='adam',
    loss=weighted_bce(pos_weights),
    metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

gen_ml = SpectreGeneratorMultiLabel13C(
    molecules_13C, df_annotations, colonnes_fonctions,
    points_13C, batch_size=128, n_per_molecule=50, augmentation=True
)

callbacks_ml = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                     patience=7, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/modele_prediction_RMN13C.keras',
        monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history_ml = model_ml.fit(
    gen_ml, epochs=30,
    validation_data=(X_val_ml, Y_val_ml),
    callbacks=callbacks_ml
)

Molécules avec annotations : 1103
Epoch 1/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 44s 87ms/step - auc: 0.5791 - loss: 0.9954 - val_auc: 0.7113 - val_loss: 0.6356 - learning_rate: 0.0010
Epoch 2/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - auc: 0.7038 - loss: 0.6285 - val_auc: 0.7831 - val_loss: 0.5620 - learning_rate: 0.0010
Epoch 3/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - auc: 0.7602 - loss: 0.5705 - val_auc: 0.8373 - val_loss: 0.4975 - learning_rate: 0.0010
Epoch 4/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 34s 77ms/step - auc: 0.8032 - loss: 0.5251 - val_auc: 0.8694 - val_loss: 0.4579 - learning_rate: 0.0010
Epoch 5/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 35s 80ms/step - auc: 0.8282 - loss: 0.4982 - val_auc: 0.8852 - val_loss: 0.4328 - learning_rate: 0.0010
Epoch 6/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 34s 77ms/step - auc: 0.8456 - loss: 0.4770 - val_auc: 0.8998 - val_loss: 0.4093 - learning_rate: 0.0010
Epoch 7/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 34s 78ms/step - auc: 0.8608 - loss: 0.4576 - val_auc: 0.9117 -

In [ ]:
import os

dossier_sauvegarde = '/content/drive/MyDrive/IA RMN/mon_modele_tf 13C'

if not os.path.exists(dossier_sauvegarde):
    os.makedirs(dossier_sauvegarde)

chemin_h5 = os.path.join(dossier_sauvegarde, 'modele_prediction_RMN13C.h5')
model_ml.save(chemin_h5)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

Y_pred_proba = model_ml.predict(X_val_ml, batch_size=128, verbose=1)

seuils_optimaux = {}
for i, fonction in enumerate(colonnes_fonctions):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 0.91, 0.05):
        f = f1_score(Y_val_ml[:, i], (Y_pred_proba[:, i] >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    seuils_optimaux[fonction] = best_t

print(f"{'Fonction':<22} {'Précision':>10} {'Rappel':>8} {'F1':>8} {'seuil':>7}  (n)")
print("-" * 68)
Y_pred_opt = np.zeros_like(Y_pred_proba, dtype=int)
for i, fonction in enumerate(colonnes_fonctions):
    t = seuils_optimaux[fonction]
    Y_pred_opt[:, i] = (Y_pred_proba[:, i] >= t).astype(int)
    p = precision_score(Y_val_ml[:, i], Y_pred_opt[:, i], zero_division=0)
    r = recall_score(Y_val_ml[:, i], Y_pred_opt[:, i], zero_division=0)
    f = f1_score(Y_val_ml[:, i], Y_pred_opt[:, i], zero_division=0)
    n = int(Y_val_ml[:, i].sum())
    print(f"  {fonction:<22} {p:>9.3f} {r:>7.3f} {f:>7.3f} {t:>6.2f}  ({n})")

f1_macro = f1_score(Y_val_ml, Y_pred_opt, average='macro', zero_division=0)
print(f"\nF1 macro (seuils optimisés) : {f1_macro:.3f}")

with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/seuils_optimaux_13C.json', 'w') as f:
    json.dump(seuils_optimaux, f, indent=2)

86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Fonction                Précision   Rappel       F1   seuil  (n)
--------------------------------------------------------------------
  aromatique                 0.931   0.959   0.945   0.50  (4962)
  alcool                     0.867   0.860   0.863   0.45  (2973)
  phenol                     0.698   0.863   0.772   0.55  (1179)
  acide_carboxylique         0.780   0.832   0.805   0.50  (3268)
  cetone                     0.584   0.636   0.609   0.45  (1523)
  aldehyde                   0.525   0.697   0.598   0.40  (1134)
  amine                      0.601   0.716   0.653   0.45  (2784)
  ester                      0.717   0.724   0.721   0.50  (1682)
  ether                      0.901   0.840   0.870   0.55  (1458)
  halogenure                 0.600   0.615   0.607   0.45  (1373)
  nitrile                    0.766   0.792   0.778   0.50  (912)
  amide                      0.571   0.715   0.635   0.45  (2031)
  heterocycle_n              0.818  

In [5]:
model_prediction = tf.keras.models.load_model(
    '/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/modele_prediction_RMN13C.keras',
    compile=False
)

with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/seuils_optimaux_13C.json') as f:
    seuils_optimaux = json.load(f)

In [33]:
def predire_fonctions_13C(csv_path, n_essais=10):
    df = pd.read_csv(csv_path)[['ppm', 'Int.']]
    df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
    df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
    df.dropna(inplace=True)
    df = normaliser_intensites(df)

    ppm_arr = df['ppm'].values
    int_arr = df['Int.'].values

    preds = []
    for _ in range(n_essais):
        X_rand  = np.random.uniform(-1.0, 1.0)
        Y_rand  = np.random.uniform(0.8, 1.2)
        largeur = np.random.uniform(0.015, 0.025)

        spectre = np.zeros_like(points_13C)
        for p, h in zip(ppm_arr, int_arr):
            spectre += crpic(points_13C, p + X_rand, h * Y_rand, largeur)

        amax = np.max(spectre)
        snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        spectre += np.random.normal(0, amax / snr if amax > 0 else 0.01, len(points_13C))

        x = spectre[np.newaxis, ..., np.newaxis].astype(np.float32)
        preds.append(model_prediction(x, training=False).numpy()[0])

    predictions = np.mean(preds, axis=0)

    print(f"Fonctions détectées ({n_essais} tirages) :")
    print("-" * 55)
    detectees = []
    for fonction, proba in zip(colonnes_fonctions, predictions):
        seuil = seuils_optimaux[fonction]
        marque = "OUI" if proba >= seuil else "non"
        print(f"  {fonction:<22} {proba*100:5.1f}%  (seuil {seuil:.2f})  {marque}")
        if proba >= seuil:
            detectees.append(fonction)

    print(f"\n→ Fonctions présentes : {', '.join(detectees) if detectees else 'aucune'}")
    return predictions

predire_fonctions_13C('/content/drive/MyDrive/IA RMN/Molecules CSV 13C test/Cyanoacetamide.csv')

Fonctions détectées (10 tirages) :
-------------------------------------------------------
  aromatique              15.5%  (seuil 0.50)  non
  alcool                  14.3%  (seuil 0.45)  non
  phenol                   4.1%  (seuil 0.55)  non
  acide_carboxylique      25.6%  (seuil 0.50)  non
  cetone                   0.0%  (seuil 0.45)  non
  aldehyde                16.3%  (seuil 0.40)  non
  amine                    7.8%  (seuil 0.45)  non
  ester                    2.3%  (seuil 0.50)  non
  ether                    0.1%  (seuil 0.55)  non
  halogenure               0.0%  (seuil 0.45)  non
  nitrile                 92.9%  (seuil 0.50)  OUI
  amide                   56.1%  (seuil 0.45)  OUI
  heterocycle_n            2.2%  (seuil 0.50)  non
  sulfoxyde                0.0%  (seuil 0.45)  non
  alcene                  19.7%  (seuil 0.55)  non

→ Fonctions présentes : nitrile, amide


array([1.5481910e-01, 1.4308408e-01, 4.0656839e-02, 2.5616637e-01,
       2.5948696e-06, 1.6301911e-01, 7.7586614e-02, 2.2862157e-02,
       1.0296401e-03, 1.5021123e-04, 9.2879593e-01, 5.6053907e-01,
       2.1973001e-02, 2.0623119e-07, 1.9728918e-01], dtype=float32)